# CASE:CONTROL MATCHING 
This script performs case-control matching using age at biospecimen collection to match controls to cases via a k-nearest neighbors approach. Additionally, multiple case-to-control matching ratios were applied to generate distinct cohorts, including 1:5, 1:10, 1:50, 1:100, and 1:300. The case-control matched cohorts generated by this script are saved to the workspace bucket.

In [ ]:
# Load packages 
library(tidyverse)
library(data.table)
library(MatchIt)

In [ ]:
# Workspace bucket 
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')
# Download phenotype file from workspace bucket 
path <- '/data/saige_phenotype_files/aml_biosample_age_files/EUR_phenotype_file.txt .'
system(paste0('gsutil cp ', my_bucket, path))
# Import cohort into workspace 
df <- fread("EUR_phenotype_file.txt")

In [ ]:
# Workspace bucket 
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

# Ratios for case:control matching 
for (r in c(5, 10, 50, 100, 300)){ 
    
    # Matching controls to cases 
    m.out <- matchit(aml_biosample_age ~ age,
                     data = df,
                     method = "nearest",
                     ratio = r,
                     calipher = 1,
                     std.caliper = TRUE)
    # Print summary 
    print(summary(m.out))
    
    # Save sample size summary 
    write.table(
      summary(m.out)[2],
      file = paste0("matched_ratio", r, "_sample_size.txt"),
      sep = "\t",
      quote = FALSE,
      col.names = NA
    )
    
    # Save statistics pre case:control matching 
    write.table(
      summary(m.out)[3],
      file = paste0("matched_ratio", r, "_prior_summary.txt"),
      sep = "\t",
      quote = FALSE,
      col.names = NA
    )
    
    # Save statistics post case:control mathcing 
    write.table(
      summary(m.out)[4],
      file = paste0("matched_ratio", r, "_post_summary.txt"),
      sep = "\t",
      quote = FALSE,
      col.names = NA
    )
    
    # Save summary statistics to workspace bucket 
    path <- paste0('matched_ratio', r,'_sample_size.txt')
    bucket <- paste0(my_bucket, '/data/matched/summary/')
    system(paste('gsutil cp', path, bucket), intern=TRUE)
    
    path <- paste0('matched_ratio', r,'_prior_summary.txt')
    system(paste('gsutil cp', path, bucket), intern=TRUE)
    
    path <- paste0('matched_ratio', r,'_post_summary.txt')
    system(paste('gsutil cp', path, bucket), intern=TRUE)
    
    # Get matched case:control cohort 
    matched <- match.data(m.out)
    
    # Save matched case:control cohort 
    write.table(
      matched,
      file = paste0("EUR_matched_ratio", r, "_cohort.txt"),
      sep = "\t",
      quote = FALSE,
      col.names = NA
    )
    
    # Save matched case:control cohort to workspace bucket 
    path <- paste0('EUR_matched_ratio', r,'_cohort.txt')
    bucket <- paste0(my_bucket, '/data/matched/cohort/')
    system(paste('gsutil cp ', path, bucket), intern=TRUE)
    
} 
